In [ ]:
import subprocess
import sys

# Install required langchain packages with compatible versions
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langchain", "langchain-core", "langchain-community", "langchain-text-splitters"])


In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import pandas as pd


inputs = [
    "What is the main contribution of the Transformer architecture?",
    "Explain the scaled dot-product attention mechanism and its formula.",
    "What are the three ways multi-head attention is applied in the Transformer?",
    "How does the Transformer handle positional information?",
    "What training hyperparameters were used for the base Transformer model?",
    "What BLEU scores did the Transformer achieve on translation tasks?",
    "Compare the computational complexity of self-attention vs recurrent layers.",
    "What is the feed-forward network architecture used in the Transformer?"
  ]
outputs = [
    "The Transformer is a new simple network architecture based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. It is the first sequence transduction model relying entirely on self-attention to compute representations of its input and output without using sequence-aligned RNNs or convolution.",
    "Scaled dot-product attention computes attention by taking dot products of the query with all keys, dividing each by √dk, and applying a softmax function to obtain the weights on the values. The formula is: Attention(Q, K, V) = softmax(QK^T / √dk)V. The scaling factor of 1/√dk counteracts the effect of large dot products pushing the softmax function into regions with extremely small gradients.",
    "The three ways are: (1) Encoder-decoder attention layers where queries come from the decoder and keys/values from the encoder output, (2) Encoder self-attention layers where all keys, values, and queries come from the previous encoder layer, and (3) Decoder self-attention layers where each position attends to all previous positions in the decoder.",
    "The Transformer uses positional encodings to inject information about the relative or absolute position of tokens in the sequence. These encodings are added to the input embeddings at the bottoms of the encoder and decoder stacks. It uses sine and cosine functions of different frequencies: PE(pos,2i) = sin(pos/10000^(2i/dmodel)) and PE(pos,2i+1) = cos(pos/10000^(2i/dmodel)).",
    "The base model was trained with: 100,000 steps (12 hours) on 8 NVIDIA P100 GPUs, Adam optimizer with β1=0.9, β2=0.98, ε=10^-9, learning rate schedule with warmup_steps=4000, dropout rate Pdrop=0.1, and label smoothing ls=0.1. The learning rate formula is: lrate = dmodel^-0.5 * min(step_num^-0.5, step_num * warmup_steps^-1.5).",
    "On the WMT 2014 English-to-German translation task, the Transformer (big) achieved a BLEU score of 28.4, outperforming previous best results by over 2 BLEU. On the WMT 2014 English-to-French task, it achieved a BLEU score of 41.0, with training taking only 3.5 days on 8 GPUs.",
    "Self-attention layers have a complexity of O(n²·d) per layer with O(1) sequential operations and O(1) maximum path length. Recurrent layers have O(n·d²) complexity with O(n) sequential operations and O(n) maximum path length. Self-attention is faster for sequences shorter than the representation dimension. The key advantage is that self-attention connects all positions with constant sequential operations.",
    "Each layer in the encoder and decoder contains a position-wise feed-forward network consisting of two linear transformations with a ReLU activation in between. The formula is: FFN(x) = max(0, xW1 + b1)W2 + b2. The input and output dimension is dmodel=512, and the inner-layer has dimension dff=2048. This can also be described as two convolutions with kernel size 1."
  ]

qa_pairs = [{"question" : q, "answers": a} for q,a in zip(inputs,outputs)]
df = pd.DataFrame(qa_pairs)

csv_path = "C:\\Users\\user\\source\\llmops\\data\\eval_dataset.csv"
df.to_csv(csv_path,index=False)

In [ ]:
from langsmith import Client

client = Client()
dataset_name = "llmops_question_answer_pair"

dataset = client.create_dataset(
    dataset_name = dataset_name,
    description="Input and expected output pairs for AgenticAIReport"
)

client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id = dataset.id,
)

In [ ]:
import sys
import os
sys.path.append("C:\\Users\\user\\source\\llmops")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG

class LocalFileAdapter:
    """Adapter for local paths to work with ChatIngestor."""
    
    def __init__(self,file_path:str):
        self.path = Path(file_path)
        self.name = self.path.name
        
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()
    
def answer_ai_report_question(
    inputs: dict,
    data_path: str = "C:\\Users\\user\\source\\llmops\\data\\NIPS-2017-attention-is-all-you-need-Paper.pdf",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
        Answer questions about the AI Engineering Report using RAG.
        Args:
            inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
            data_path: Path to the AI Engineering Report text file
            chunk_size: Size of text chunks for splitting
            chunk_overlap: Overlap between chunks
            k: Number of documents to retrieve
    """
    try:
        
        question = inputs.get("question","")
        if not question:
            return {"answer": "No Question provided"}
        
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        file_adapter = LocalFileAdapter(data_path)   
        
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        ) 
        
        ingestor.build_text_retriever(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME","index")
        )
        
        answer = rag.invoke(question,chat_history=[])
        return {"answer": answer}
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}

In [ ]:
test_input = {"question": "What is the main contribution of the Transformer architecture?"}
result = answer_ai_report_question(test_input)
print("Question:",test_input["question"])
print("\nAnswer:",result["answer"])

In [ ]:
from langsmith.evaluation import evaluate, 

In [ ]:
print("Testing all questions from the dataset: \n")
for i,q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-"*80+"\n")    

In [ ]:
from langsmith.evaluation import evaluate
from langchain_classic.evaluation import load_evaluator
from langchain_groq import  ChatGroq

llm = ChatGroq(
                model = "llama-3.3-70b-versatile",
                api_key = os.getenv("GROQ_API_KEY"),
            )
qa_evaluator_chain = load_evaluator("cot_qa", llm=llm)

def qa_evaluator(run,example):
    result = qa_evaluator_chain.evaluate_strings(
        prediction=run.outputs["answer"],  # <-- FIXED: use 'answer' not 'output'
        reference=example.outputs["answer"],
        input=example.inputs["question"]
    )
    return {"key": "cot_qa","score": result["score"]}

dataset_name = "llmops_question_answer_pair"

experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=[qa_evaluator],
    experiment_prefix="test-agenticAIReport-qa-rag",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)